In [1]:
import numpy as np
import matplotlib.pyplot as plt

In [2]:
class Missile:
    def __init__(self, position, velocity) -> None:
        self.position = np.array(position, dtype=float)
        self.velocity = np.array(velocity, dtype=float)
        self.path = [self.position.copy()]
    
    def move(self, dt=1.0):
        self.position += self.velocity * dt
        self.path.append(self.position.copy())

In [3]:
class Interceptor:
    def __init__(self, position, speed) -> None:
        self.position = np.array(position, dtype=float)
        self.speed = speed
        self.path = [self.position.copy()]
        self.active = True
    
    def move_towards(self, target_pos, dt=1.0):
        if not self.active:
            return
        direction = target_pos - self.position
        distance = np.linalg.norm(direction)
        if distance < 1e-6:
            self.active = False
            return
        direction = direction / distance
        step = min(self.speed * dt, distance)
        self.position += direction*step
        self.path.append(self.position.copy())

In [4]:
def simulate(missiles, interceptors, dt=0.1, max_time=20.0, intercept_radius=5.0):
    time = 0.0
    while time < max_time:
        for missile in missiles:
            missile.move(dt)
        
        for interceptor in interceptors:
            if interceptor.active:
                interceptor.move_towards(missiles[0].position, dt)
        
        for missile in missiles:
            for interceptor in interceptors:
                if interceptor.active and np.linalg.norm(missile.position - interceptor.position) < intercept_radius:
                    print(f"INTERCEPTED AT TIME: {time:.3f}s | POSITION: {missile.position}")
                    interceptor.active = False
        
        time += dt